# 06 — Explainability (SHAP)

- Summary plot (bar + beeswarm) — глобальная важность
- Waterfall plot — одно предсказание
- Group contributions (technical / fundamental / news)
- Dependence plots — топ-3 признака
- Сравнение SHAP-важности между моделями

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'backend'))

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR    = ROOT / 'models'
TICKERS       = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META', 'JPM', 'JNJ', 'V']
FOCUS_TICKER  = 'AAPL'
FOCUS_MODEL   = 'model_a'
TRAIN_RATIO   = 0.8

print(f'shap: {shap.__version__}')

## 1. Загрузка модели и данных

In [ ]:
from app.models.baseline_lr import BaselineLRModel
from app.models.model_a import ModelA
from app.models.model_b import ModelB
from app.models.model_c import ModelC
from app.models.model_d import ModelD
from app.features.constants import TECHNICAL_FEATURES, LAG_FEATURES, FUNDAMENTAL_FEATURES, SENTIMENT_FEATURES

model_classes = {
    'baseline_lr': BaselineLRModel,
    'model_a':     ModelA,
    'model_b':     ModelB,
    'model_c':     ModelC,
    'model_d':     ModelD,
}


def load_model(name: str):
    artifact = MODELS_DIR / f'{name}.joblib'
    if not artifact.exists():
        raise FileNotFoundError(f'Артефакт {artifact} не найден — запустите ноутбук 04')
    m = model_classes[name]()
    m.load(artifact)
    return m


def load_data(ticker: str) -> pd.DataFrame:
    path = PROCESSED_DIR / ticker / 'technical.parquet'
    if not path.exists():
        raise FileNotFoundError(f'Нет данных для {ticker}')
    df = pd.read_parquet(path)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)


model  = load_model(FOCUS_MODEL)
df     = load_data(FOCUS_TICKER)

features = model.feature_set
avail    = [f for f in features if f in df.columns]
X        = df[avail].fillna(0)

split_idx = int(len(X) * TRAIN_RATIO)
X_test    = X.iloc[split_idx:]

print(f'Модель: {FOCUS_MODEL}  |  Тикер: {FOCUS_TICKER}')
print(f'Признаков: {len(avail)}  |  Тест: {len(X_test)} строк')

## 2. SHAP — глобальный анализ

In [ ]:
from app.explainability.shap_explainer import ShapExplainer

explainer   = ShapExplainer(model.model, feature_names=avail)
shap_values = explainer.explain_batch(X_test)

print(f'SHAP values shape: {shap_values.shape}')

shap.summary_plot(
    shap_values, X_test,
    plot_type='bar', feature_names=avail,
    show=True, plot_size=(10, 6),
)

In [ ]:
shap.summary_plot(
    shap_values, X_test,
    feature_names=avail,
    show=True, plot_size=(10, 7),
)

## 3. Waterfall — одно предсказание

In [ ]:
SAMPLE_IDX = -1

single   = explainer.explain_single(X_test.iloc[[SAMPLE_IDX]])
base_val = explainer.base_value

shap_exp = shap.Explanation(
    values       = single,
    base_values  = base_val,
    data         = X_test.iloc[SAMPLE_IDX].values,
    feature_names= avail,
)
shap.plots.waterfall(shap_exp, max_display=20, show=True)

## 4. Group Contributions

In [ ]:
groups = {
    'Technical': set(TECHNICAL_FEATURES) | set(LAG_FEATURES),
    'Fundamental': set(FUNDAMENTAL_FEATURES),
    'News/Sentiment': set(SENTIMENT_FEATURES),
}

avg_abs  = np.abs(shap_values).mean(axis=0)
feat_idx = {f: i for i, f in enumerate(avail)}
total    = avg_abs.sum()

scores = {}
for group, feat_set in groups.items():
    idxs = [feat_idx[f] for f in feat_set if f in feat_idx]
    scores[group] = avg_abs[idxs].sum() / total if total > 0 else 0

print('Group Contributions (avg |SHAP|):')
for g, v in scores.items():
    print(f'  {g:20s}: {v:.2%}')

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ['#6366f1', '#22c55e', '#f59e0b']
bars = ax.bar(scores.keys(), scores.values(), color=bar_colors, width=0.5)
ax.set_ylabel('Доля avg |SHAP|')
ax.set_title(f'Group Contributions — {FOCUS_MODEL} / {FOCUS_TICKER}')
ax.set_ylim(0, 1)
for bar, val in zip(bars, scores.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Dependence plots — топ-3 признака

In [ ]:
top3 = np.argsort(avg_abs)[::-1][:3]

for idx in top3:
    fname = avail[idx]
    shap.dependence_plot(
        idx, shap_values, X_test,
        feature_names=avail,
        title=f'Dependence: {fname}',
        show=True,
    )

## 6. Сравнение SHAP-важности: baseline_lr vs model_a

In [ ]:
compare = ['baseline_lr', 'model_a']
shap_imp = {}

for mname in compare:
    try:
        m = load_model(mname)
        fts  = m.feature_set
        av   = [f for f in fts if f in df.columns]
        Xm   = df[av].fillna(0).iloc[split_idx:]
        exp  = ShapExplainer(m.model, feature_names=av)
        sv   = exp.explain_batch(Xm)
        shap_imp[mname] = pd.Series(np.abs(sv).mean(axis=0), index=av)
        print(f'[OK] {mname}')
    except Exception as e:
        print(f'[SKIP] {mname}: {e}')

if len(shap_imp) >= 2:
    comp_df = pd.DataFrame(shap_imp).fillna(0)
    top_feats = comp_df.mean(axis=1).nlargest(15).index

    comp_df.loc[top_feats].plot(
        kind='barh', figsize=(10, 7),
        color=['#6366f1', '#22c55e'],
    )
    plt.xlabel('avg |SHAP|')
    plt.title('Сравнение SHAP-важности — baseline_lr vs model_a')
    plt.tight_layout()
    plt.show()